In [3]:
# ── CELL 1: LOAD ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

SAVE_DIR = '/content/drive/MyDrive/isro_hackathon_data'

goes   = pd.read_csv(f'{SAVE_DIR}/goes_preprocessed.csv')
goes['time'] = pd.to_datetime(goes['time'], format='mixed')
goes   = goes.sort_values('time').reset_index(drop=True)

flares = pd.read_csv(f'{SAVE_DIR}/flares_cache.csv', parse_dates=['start_time'])
mx_flares = flares[flares['class'].str[0].isin(['M', 'X'])].copy()

print(f"Rows: {len(goes):,}")
print(f"Date range: {goes['time'].min()} to {goes['time'].max()}")
print(f"Columns: {goes.columns.tolist()}")
print(f"Total reference flares: {len(flares)}")
print(f"M/X reference flares: {len(mx_flares)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows: 7,318,564
Date range: 2010-04-07 19:45:00 to 2024-12-31 23:59:00
Columns: ['time', 'segment_id', 'log_soft', 'log_hard', 'log_hardening', 'dFs_dt', 'dFh_dt', 'Fs_ma5', 'd2Fs_dt2']
Total reference flares: 22923
M/X reference flares: 1311


In [4]:
# ── PARAMETERS ────────────────────────────────────────────────────
WINDOW_MIN      = 30    # rolling baseline window
LAG_MIN         = 10    # lag so flare rise doesn't inflate baseline
SIGMA_THRESHOLD = 3.0   # optimal from ROC curve
COINCIDENCE_MIN = 2     # dual-channel coincidence window
MIN_STD         = 0.01  # guard against flat-signal divide-by-zero
Z_CLIP          = 1000  # clip artifact z-scores

# ── STEP 1 & 2: Lagged baseline + z-score per segment ─────────────
def compute_lagged_baseline(df):
    result = df.copy()
    for seg_id, group in df.groupby('segment_id'):
        idx  = group.index
        soft = group['log_soft']
        hard = group['log_hard']
        result.loc[idx, 'baseline_soft'] = soft.shift(LAG_MIN).rolling(window=WINDOW_MIN, min_periods=10).mean()
        result.loc[idx, 'std_soft']      = soft.shift(LAG_MIN).rolling(window=WINDOW_MIN, min_periods=10).std()
        result.loc[idx, 'baseline_hard'] = hard.shift(LAG_MIN).rolling(window=WINDOW_MIN, min_periods=10).mean()
        result.loc[idx, 'std_hard']      = hard.shift(LAG_MIN).rolling(window=WINDOW_MIN, min_periods=10).std()
    return result

print("Computing lagged baseline per segment...")
goes = compute_lagged_baseline(goes)
goes = goes.dropna(subset=['baseline_soft','std_soft','baseline_hard','std_hard']).reset_index(drop=True)
print(f"Rows after warmup drop: {len(goes)}")

goes['zscore_soft'] = (goes['log_soft'] - goes['baseline_soft']) / goes['std_soft'].clip(lower=MIN_STD)
goes['zscore_hard'] = (goes['log_hard'] - goes['baseline_hard']) / goes['std_hard'].clip(lower=MIN_STD)
goes['zscore_soft'] = goes['zscore_soft'].clip(-Z_CLIP, Z_CLIP)
goes['zscore_hard'] = goes['zscore_hard'].clip(-Z_CLIP, Z_CLIP)

print(f"zscore_soft — max: {goes['zscore_soft'].max():.1f} | 99th pct: {goes['zscore_soft'].quantile(0.99):.2f}")
print(f"zscore_hard — max: {goes['zscore_hard'].max():.1f} | 99th pct: {goes['zscore_hard'].quantile(0.99):.2f}")

Computing lagged baseline per segment...
Rows after warmup drop: 7276680
zscore_soft — max: 270.6 | 99th pct: 14.14
zscore_hard — max: 356.2 | 99th pct: 11.21


In [5]:
# ── STEP 3: Candidate Flagging ────────────────────────────────────
goes['candidate'] = (
    (goes['zscore_soft'] > SIGMA_THRESHOLD) |
    (goes['zscore_hard'] > SIGMA_THRESHOLD)
)
print(f"Step 3 done. Candidates: {goes['candidate'].sum()} rows ({goes['candidate'].mean()*100:.3f}%)")

# ── STEP 4: Dual-Channel Coincidence Confirmation ─────────────────
coin_window = 2 * COINCIDENCE_MIN + 1
soft_above  = goes['zscore_soft'] > SIGMA_THRESHOLD
hard_above  = goes['zscore_hard'] > SIGMA_THRESHOLD
soft_any    = soft_above.rolling(window=coin_window, center=True, min_periods=1).max().astype(bool)
hard_any    = hard_above.rolling(window=coin_window, center=True, min_periods=1).max().astype(bool)

goes['confirmed_flare'] = soft_any & hard_any

n_candidates = goes['candidate'].sum()
n_confirmed  = goes['confirmed_flare'].sum()
rejection    = (1 - n_confirmed / n_candidates) * 100 if n_candidates > 0 else 0

print(f"Step 4 done.")
print(f"  Candidates : {n_candidates}")
print(f"  Confirmed  : {n_confirmed}")
print(f"  Rejected by dual-channel filter: {rejection:.1f}%")

Step 3 done. Candidates: 813463 rows (11.179%)
Step 4 done.
  Candidates : 813463
  Confirmed  : 526028
  Rejected by dual-channel filter: 35.3%


In [6]:
# ── STEPS 5-8: VECTORIZED EVENT GROUPING + CLASSIFICATION + MASTER CATALOGUE ──
import gc

GOES_THRESHOLDS = [('X',1e-4),('M',1e-5),('C',1e-6),('B',1e-7),('A',0.0)]

def classify_flux(peak_flux):
    for cls, thresh in GOES_THRESHOLDS:
        if peak_flux >= thresh:
            divisor = thresh if thresh > 0 else 1e-8
            return f"{cls}{peak_flux/divisor:.1f}"
    return 'A0.0'

# Step 5 — Event grouping (vectorized)
goes['event_block'] = (goes['confirmed_flare'] != goes['confirmed_flare'].shift()).cumsum()

# Work only on confirmed rows — tiny subset of 7M
confirmed = goes[goes['confirmed_flare']][
    ['time', 'event_block', 'log_soft', 'zscore_soft']
].copy()
confirmed['raw_soft'] = 10 ** confirmed['log_soft']

# Step 6 — Vectorized peak finding per block
agg = confirmed.groupby('event_block').agg(
    start_time   = ('time', 'first'),
    end_time     = ('time', 'last'),
    duration_min = ('time', 'count'),
    peak_flux    = ('raw_soft', 'max'),
    snr          = ('zscore_soft', 'max'),
)

# Get peak time — time at max zscore per block
peak_idx_per_block = confirmed.groupby('event_block')['zscore_soft'].idxmax()
agg['peak_time'] = confirmed.loc[peak_idx_per_block, 'time'].values

del confirmed
gc.collect()

# Step 7 — Drop bad SNR
agg = agg[agg['snr'] > 0].reset_index(drop=True)

# Step 8 — Classify and build master catalogue
agg['flare_class'] = agg['peak_flux'].apply(classify_flux)

master_catalogue = pd.DataFrame({
    'event_id'          : [f"FL_{i+1:04d}" for i in range(len(agg))],
    'start_time'        : agg['start_time'].values,
    'peak_time'         : agg['peak_time'].values,
    'end_time'          : agg['end_time'].values,
    'duration_min'      : agg['duration_min'].values,
    'flare_class'       : agg['flare_class'].values,
    'peak_flux'         : agg['peak_flux'].values,
    'snr'               : agg['snr'].values,
    'confirmed_channels': 'soft+hard',
})

master_catalogue = master_catalogue.dropna(subset=['peak_flux','snr']).reset_index(drop=True)
master_catalogue.to_csv(f'{SAVE_DIR}/master_catalogue.csv', index=False)

print(f"Step 5-8 done. {len(master_catalogue):,} events")
print(f"\nClass distribution:")
print(master_catalogue['flare_class'].str[0].value_counts())
print(f"\nSNR — mean: {agg['snr'].mean():.1f} | max: {agg['snr'].max():.1f} | min: {agg['snr'].min():.1f}")
print(f"\nFirst 5 events:")
print(master_catalogue.head(5).to_string(index=False))

Step 5-8 done. 45,684 events

Class distribution:
flare_class
C    24623
B    16303
A     2436
M     2190
X      132
Name: count, dtype: int64

SNR — mean: 12.2 | max: 270.6 | min: 0.0

First 5 events:
event_id          start_time           peak_time            end_time  duration_min flare_class    peak_flux      snr confirmed_channels
 FL_0001 2010-04-07 23:56:00 2010-04-07 23:56:00 2010-04-07 23:56:00             1        A6.1 6.111696e-08 2.551820          soft+hard
 FL_0002 2010-04-07 23:58:00 2010-04-07 23:59:00 2010-04-08 00:03:00             6        A8.0 8.006462e-08 3.904362          soft+hard
 FL_0003 2010-04-08 02:35:00 2010-04-08 02:44:00 2010-04-08 02:52:00            18        B1.9 1.902465e-07 8.015093          soft+hard
 FL_0004 2010-04-09 04:44:00 2010-04-09 04:52:00 2010-04-09 05:00:00            17        B5.5 5.549241e-07 8.777621          soft+hard
 FL_0005 2010-04-13 04:40:00 2010-04-13 04:43:00 2010-04-13 04:53:00            14        B5.5 5.493436e-07 5.974589  

In [7]:
# ── STEP 9: Validation (M/X only, 30-min window) ─────────────────
MATCH_NS_30 = 30 * 60 * 1_000_000_000

t_min = master_catalogue['start_time'].min()
t_max = master_catalogue['end_time'].max()
ref   = flares[(flares['start_time'] >= t_min) & (flares['start_time'] <= t_max)].copy()
total_minutes = int((t_max - t_min).total_seconds() / 60)

mx_ref    = ref[ref['class'].str[0].isin(['M', 'X'])].copy()
mx_ref_ns = mx_ref['start_time'].values.astype('int64')
det_ns    = master_catalogue['start_time'].values.astype('int64')
det_ns_sorted = np.sort(det_ns)

tp_ = 0
for ref_t in mx_ref_ns:
    lo = ref_t - MATCH_NS_30
    hi = ref_t + MATCH_NS_30
    if np.searchsorted(det_ns_sorted, hi, side='right') > np.searchsorted(det_ns_sorted, lo, side='left'):
        tp_ += 1

fp_ = 0
mx_ref_ns_sorted = np.sort(mx_ref_ns)
for det_t in det_ns:
    lo = det_t - MATCH_NS_30
    hi = det_t + MATCH_NS_30
    if np.searchsorted(mx_ref_ns_sorted, hi, side='right') == np.searchsorted(mx_ref_ns_sorted, lo, side='left'):
        fp_ += 1

fn_  = len(mx_ref_ns) - tp_
tn_  = max(0, total_minutes - len(mx_ref_ns) * 30 - fp_)
TPR  = tp_ / len(mx_ref_ns) if len(mx_ref_ns) > 0 else 0
FAR  = fp_ / (fp_ + tn_)    if (fp_ + tn_) > 0    else 0
TSS  = TPR - FAR

print(f"=== FINAL VALIDATION (M/X, 30-min window) ===")
print(f"Total M/X reference flares : {len(mx_ref):,}")
print(f"TP: {tp_}  FN: {fn_}  FP: {fp_}  TN: {tn_}")
print(f"TPR : {TPR:.4f}")
print(f"FAR : {FAR:.6f}")
print(f"TSS : {TSS:.4f}")

=== FINAL VALIDATION (M/X, 30-min window) ===
Total M/X reference flares : 1,296
TP: 1233  FN: 63  FP: 44235  TN: 7666867
TPR : 0.9514
FAR : 0.005737
TSS : 0.9457


In [8]:
# ── STEP 10: ROC Curve (memory-safe, M/X only) ───────────────────
print("Step 10: Running ROC curve (M/X only, 30-min window)...")

MATCH_NS_30 = 30 * 60 * 1_000_000_000
sigma_range = np.arange(3.0, 12.0, 0.5)
roc_results = []
coin        = 2 * COINCIDENCE_MIN + 1
zs          = goes['zscore_soft'].values
zh          = goes['zscore_hard'].values
kernel      = np.ones(coin, dtype=np.int8)
times_ns    = goes['time'].values.astype('int64')
mx_ref_ns   = mx_ref['start_time'].values.astype('int64')
total_min   = total_minutes

for sigma in sigma_range:
    soft_any  = np.convolve((zs > sigma).astype(np.int8), kernel, mode='same') > 0
    hard_any  = np.convolve((zh > sigma).astype(np.int8), kernel, mode='same') > 0
    conf      = soft_any & hard_any
    padded    = np.concatenate([[False], conf])
    rising    = (~padded[:-1]) & padded[1:]
    start_ns  = times_ns[rising]
    n_det     = len(start_ns)

    if n_det == 0:
        roc_results.append({'sigma': sigma, 'TPR': 0.0, 'FAR': 0.0, 'TSS': 0.0})
        continue

    # Match each reference flare to nearest detection using searchsorted
    # Much more memory efficient than full matrix
    start_ns_sorted = np.sort(start_ns)
    tp_ = 0
    for ref_t in mx_ref_ns:
        lo = ref_t - MATCH_NS_30
        hi = ref_t + MATCH_NS_30
        left  = np.searchsorted(start_ns_sorted, lo, side='left')
        right = np.searchsorted(start_ns_sorted, hi, side='right')
        if right > left:
            tp_ += 1

    # Count false positives — detections with no nearby reference flare
    fp_ = 0
    for det_t in start_ns:
        lo = det_t - MATCH_NS_30
        hi = det_t + MATCH_NS_30
        left  = np.searchsorted(mx_ref_ns, lo, side='left')
        right = np.searchsorted(mx_ref_ns, hi, side='right')
        if right == left:
            fp_ += 1

    fn_  = len(mx_ref_ns) - tp_
    tn_  = max(0, total_min - len(mx_ref_ns) * 30 - fp_)
    tpr_ = tp_ / len(mx_ref_ns) if len(mx_ref_ns) > 0 else 0
    far_ = fp_ / (fp_ + tn_)    if (fp_ + tn_) > 0    else 0
    tss_ = tpr_ - far_
    roc_results.append({'sigma': sigma, 'TPR': tpr_, 'FAR': far_, 'TSS': tss_})
    print(f"  σ={sigma:.1f} → TPR={tpr_:.3f}  FAR={far_:.5f}  TSS={tss_:.3f}  det={n_det}")

roc_df       = pd.DataFrame(roc_results)
roc_df['dist'] = np.sqrt((1 - roc_df['TPR'])**2 + roc_df['FAR']**2)
best         = roc_df.loc[roc_df['TSS'].idxmax()]
print(f"\n✅ Optimal sigma = {best['sigma']:.1f} | TPR={best['TPR']:.3f} | FAR={best['FAR']:.5f} | TSS={best['TSS']:.3f}")

plt.figure(figsize=(7, 6))
plt.plot(roc_df['FAR'], roc_df['TPR'], 'b-o', lw=2, label='ROC curve')
plt.plot(best['FAR'], best['TPR'], 'r*', markersize=16, label=f"Optimal σ={best['sigma']:.1f}")
for _, row in roc_df.iterrows():
    plt.annotate(f"σ={row['sigma']:.1f}", (row['FAR'], row['TPR']),
                 textcoords='offset points', xytext=(6,-4), fontsize=8)
plt.xlabel('False Alarm Rate (FAR)')
plt.ylabel('True Positive Rate (TSS-optimal)')
plt.title('StatFlare ROC Curve — M/X Flares, 30-min window')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/roc_curve.png', dpi=150)
plt.show()
print("ROC curve saved to Drive")

Step 10: Running ROC curve (M/X only, 30-min window)...
  σ=3.0 → TPR=0.951  FAR=0.00575  TSS=0.946  det=45761
  σ=3.5 → TPR=0.944  FAR=0.00476  TSS=0.940  det=38116
  σ=4.0 → TPR=0.926  FAR=0.00406  TSS=0.922  det=32648
  σ=4.5 → TPR=0.903  FAR=0.00352  TSS=0.899  det=28415
  σ=5.0 → TPR=0.880  FAR=0.00307  TSS=0.877  det=24876
  σ=5.5 → TPR=0.858  FAR=0.00271  TSS=0.855  det=22066
  σ=6.0 → TPR=0.831  FAR=0.00241  TSS=0.829  det=19695
  σ=6.5 → TPR=0.799  FAR=0.00215  TSS=0.796  det=17637
  σ=7.0 → TPR=0.779  FAR=0.00192  TSS=0.777  det=15872
  σ=7.5 → TPR=0.748  FAR=0.00174  TSS=0.747  det=14425
  σ=8.0 → TPR=0.711  FAR=0.00157  TSS=0.710  det=13018
  σ=8.5 → TPR=0.682  FAR=0.00142  TSS=0.681  det=11849
  σ=9.0 → TPR=0.658  FAR=0.00130  TSS=0.657  det=10857
  σ=9.5 → TPR=0.630  FAR=0.00118  TSS=0.628  det=9950
  σ=10.0 → TPR=0.593  FAR=0.00109  TSS=0.592  det=9158
  σ=10.5 → TPR=0.573  FAR=0.00099  TSS=0.572  det=8402
  σ=11.0 → TPR=0.542  FAR=0.00092  TSS=0.541  det=7754
  σ=11.5 →

In [9]:
# ── STEP 11: Visualization ────────────────────────────────────────
os.makedirs(f'{SAVE_DIR}/flare_plots', exist_ok=True)
CONTEXT_MIN = 60

def plot_flare(row):
    start = pd.to_datetime(row['start_time'])
    end   = pd.to_datetime(row['end_time'])
    peak  = pd.to_datetime(row['peak_time'])
    t_from = start - pd.Timedelta(f'{CONTEXT_MIN}min')
    t_to   = end   + pd.Timedelta(f'{CONTEXT_MIN}min')

    seg = goes[(goes['time'] >= t_from) & (goes['time'] <= t_to)].copy()
    if len(seg) < 5:
        return None

    raw_soft    = 10 ** seg['log_soft']
    raw_hard    = 10 ** seg['log_hard']
    baseline_s  = 10 ** seg['baseline_soft']
    baseline_h  = 10 ** seg['baseline_hard']
    thresh_soft = 10 ** (seg['baseline_soft'] + SIGMA_THRESHOLD * seg['std_soft'])
    thresh_hard = 10 ** (seg['baseline_hard'] + SIGMA_THRESHOLD * seg['std_hard'])

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
    fig.suptitle(
        f"{row['event_id']}  |  Class: {row['flare_class']}  |  "
        f"Peak: {row['peak_flux']:.2e} W/m²  |  SNR: {row['snr']:.1f}σ  |  "
        f"Duration: {row['duration_min']} min",
        fontsize=11, fontweight='bold'
    )
    for ax, raw, base, thresh, color, label in [
        (ax1, raw_soft, baseline_s, thresh_soft, 'royalblue', 'Soft (1–8 Å)'),
        (ax2, raw_hard, baseline_h, thresh_hard, 'firebrick', 'Hard (0.5–4 Å)'),
    ]:
        ax.semilogy(seg['time'], raw,    color=color, lw=1.2, label=label)
        ax.semilogy(seg['time'], base,   color=color, lw=1.0, ls='--', label='Baseline')
        ax.semilogy(seg['time'], thresh, color=color, lw=1.0, ls=':',  label=f'{SIGMA_THRESHOLD}σ threshold')
        ax.axvspan(start, end, color='gold', alpha=0.3, label='Flare window')
        ax.axvline(peak,  color='red',  lw=1.5, ls='--', label='Peak')
        for lbl, val in [('X',1e-4),('M',1e-5),('C',1e-6),('B',1e-7)]:
            ax.axhline(val, color='gray', lw=0.6, ls=':', alpha=0.5)
            ax.text(seg['time'].iloc[-1], val*1.2, lbl, color='gray', fontsize=8)
        ax.set_ylabel('Flux (W/m²)', fontsize=9)
        ax.legend(fontsize=8, loc='upper left', ncol=2)
        ax.grid(True, which='both', alpha=0.2)

    peak_str = peak.strftime('%H:%M:%S') if pd.notnull(peak) else 'N/A'
    info = (f"Class:    {row['flare_class']}\n"
            f"Peak:     {row['peak_flux']:.2e} W/m²\n"
            f"SNR:      {row['snr']:.1f}σ\n"
            f"Duration: {row['duration_min']} min\n"
            f"Start:    {start.strftime('%Y-%m-%d %H:%M')}\n"
            f"Peak:     {peak_str}\n"
            f"End:      {end.strftime('%H:%M')}")
    ax1.text(0.99, 0.97, info, transform=ax1.transAxes, fontsize=8.5,
             va='top', ha='right',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    ax2.set_xlabel('Time (UTC)', fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/flare_plots/{row['event_id']}.png", dpi=120, bbox_inches='tight')
    plt.close()
    return True

n      = min(50, len(master_catalogue))
failed = 0
for i, row in master_catalogue.head(n).iterrows():
    result = plot_flare(row)
    if result is None:
        failed += 1
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{n} plots done...")

print(f"\n✅ {n-failed} plots saved to flare_plots/")
print(f"\n{'='*50}")
print(f"NOWCASTING PIPELINE COMPLETE")
print(f"  Total events     : {len(master_catalogue)}")
print(f"  TPR (overall)    : {TPR:.4f}")
print(f"  FAR              : {FAR:.6f}")
print(f"  Optimal sigma    : {best['sigma']:.1f}")
print(f"  Plots saved      : {n-failed}")
print(f"  Files            : master_catalogue.csv | roc_curve.png | flare_plots/")
print(f"{'='*50}")

  10/50 plots done...
  20/50 plots done...
  30/50 plots done...
  40/50 plots done...
  50/50 plots done...

✅ 50 plots saved to flare_plots/

NOWCASTING PIPELINE COMPLETE
  Total events     : 45684
  TPR (overall)    : 0.9514
  FAR              : 0.005737
  Optimal sigma    : 3.0
  Plots saved      : 50
  Files            : master_catalogue.csv | roc_curve.png | flare_plots/


In [10]:
print("=== DATASET DIAGNOSTIC ===")
print(f"Total rows: {len(goes)}")
print(f"Date range: {goes['time'].min()} → {goes['time'].max()}")
print(f"Unique segments: {goes['segment_id'].nunique()}")

print(f"\nZ-score stats:")
print(f"  zscore_soft — max: {goes['zscore_soft'].max():.1f} | 99th pct: {goes['zscore_soft'].quantile(0.99):.2f} | 95th pct: {goes['zscore_soft'].quantile(0.95):.2f}")
print(f"  zscore_hard — max: {goes['zscore_hard'].max():.1f} | 99th pct: {goes['zscore_hard'].quantile(0.99):.2f} | 95th pct: {goes['zscore_hard'].quantile(0.95):.2f}")

print(f"\nReference flares in window: {len(ref)}")
print(f"Class distribution in ref:")
print(ref['class'].str[0].value_counts().to_string())

known_xflares = ['2011-03-09', '2011-02-15', '2012-03-07', '2013-05-13']
print(f"\nKnown X-flare z-scores:")
for date in known_xflares:
    day = goes[goes['time'].dt.date.astype(str) == date]
    if len(day) > 0:
        peak_z    = day['zscore_soft'].max()
        peak_flux = 10 ** day['log_soft'].max()
        print(f"  {date}: max z={peak_z:.1f} | peak flux={peak_flux:.2e}")
    else:
        print(f"  {date}: not in dataset")

=== DATASET DIAGNOSTIC ===
Total rows: 7276680
Date range: 2010-04-07 20:04:00 → 2024-12-31 23:59:00
Unique segments: 2146

Z-score stats:
  zscore_soft — max: 270.6 | 99th pct: 14.14 | 95th pct: 5.02
  zscore_hard — max: 356.2 | 99th pct: 11.21 | 95th pct: 3.51

Reference flares in window: 22495
Class distribution in ref:
class
C    12550
B     8556
M     1228
A       93
X       68

Known X-flare z-scores:
  2011-03-09: max z=33.7 | peak flux=2.24e-04
  2011-02-15: max z=27.6 | peak flux=3.30e-04
  2012-03-07: max z=16.3 | peak flux=7.79e-04
  2013-05-13: max z=35.9 | peak flux=4.11e-04
